# Ingest DEG's conditional-essential pathogens -> test the rogue-zone thesis

DEG has 2,506 conditional-essential genes (M. tuberculosis cholesterol/bile, P. aeruginosa, Francisella, Salmonella, Acinetobacter) -- but those organisms aren't in our orthology, so we can't map them to conservation. This notebook ingests them: download the pathogen genomes, DIAMOND their proteins onto our OG-representative DB (no orthology recompute), attach each gene's conservation, then run the decisive test:

**Are DEG's CONDITIONAL essentials lower-conservation (rogue-zone) than its UNCONDITIONAL essentials?**

If yes -> conditional essentiality IS the rogue zone, confirming the project thesis on independent pathogen data.
If no (e.g. conditional genes are housekeeping like dnaK/ileS) -> thesis is wrong / more nuanced. Honest either way.

Reference (committed in repo): `og_rep_proteins.faa` (7,517 OG families) + `og_conservation.json`. GPU not needed; DIAMOND is CPU.

## 1. Setup + build OG DIAMOND DB

In [ ]:
!pip install -q ncbi-datasets-cli pandas
!wget -q https://github.com/bbuchfink/diamond/releases/download/v2.1.9/diamond-linux64.tar.gz -O /tmp/d.tgz
!tar xzf /tmp/d.tgz -C /tmp && chmod +x /tmp/diamond
!git clone --depth 1 -b claude/vectorize-gex-propensity-NRqBW https://github.com/nikku03/cell.git cell_repo || echo cloned
import os; os.chdir('cell_repo')
!/tmp/diamond makedb --in outputs/orphan/og_rep_proteins.faa -d /tmp/ogdb --quiet
!ls -lh /tmp/ogdb.dmnd

## 2. Download the conditional-essential pathogen genomes (proteins + GFF)

In [ ]:
# DEG organism substring -> RefSeq assembly accession
PATHOGENS = {
  'Mycobacterium tuberculosis H37Rv': 'GCF_000195955.2',
  'Pseudomonas aeruginosa PAO1':      'GCF_000006765.1',
  'Francisella tularensis':           'GCF_000008985.1',
  'Salmonella enterica serovar Typhi':'GCF_000195995.1',
  'Acinetobacter baumannii ATCC 17978':'GCF_000015425.1',
  'Salmonella enterica subsp. enterica serovar Typhimurium': 'GCF_000022165.1',
}
accs = list(set(PATHOGENS.values()))
open('/tmp/path_accs.txt','w').write('\n'.join(accs)+'\n')
!datasets download genome accession --inputfile /tmp/path_accs.txt --include protein,gff3 --filename /tmp/path.zip
!unzip -o -q /tmp/path.zip -d /tmp/path
import glob; print('downloaded:', glob.glob('/tmp/path/ncbi_dataset/data/GCF_*'))

## 3. DIAMOND pathogen proteins -> our OGs

In [ ]:
import glob, os
# concat all pathogen proteins, tag headers with accession
q=open('/tmp/path_q.faa','w')
for fp in glob.glob('/tmp/path/ncbi_dataset/data/GCF_*/protein.faa'):
    acc=fp.split('/')[-2]
    for line in open(fp):
        if line.startswith('>'): q.write('>'+acc+'|'+line[1:].split()[0]+'\n')
        else: q.write(line)
q.close()
!/tmp/diamond blastp -q /tmp/path_q.faa -d /tmp/ogdb -o /tmp/path_hits.tsv \
  --outfmt 6 qseqid sseqid pident --id 30 --query-cover 50 --max-target-seqs 1 \
  --threads 4 --quiet
!wc -l /tmp/path_hits.tsv
# best OG per pathogen protein (acc|protein_id -> og)
import csv
prot2og={}
for line in open('/tmp/path_hits.tsv'):
    q_,og,pid=line.split('\t'); prot2og.setdefault(q_, og)   # first = best (max-target-seqs 1)
print('pathogen proteins mapped to an OG:', len(prot2og))

## 4. Bridge: pathogen gene_name -> locus_tag -> protein_id -> OG (per genome GFF)

In [ ]:
import re, glob
# per accession: gene_name->protein_id  AND locus_tag->protein_id
acc_gene2prot={}; acc_locus2prot={}
for gff in glob.glob('/tmp/path/ncbi_dataset/data/GCF_*/genomic.gff'):
    acc=gff.split('/')[-2]; g2p={}; l2p={}
    for line in open(gff):
        if '\tCDS\t' not in line: continue
        gn=re.search(r'gene=([^;\n]+)',line); pid=re.search(r'protein_id=([^;\n]+)',line); lt=re.search(r'locus_tag=([^;\n]+)',line)
        if pid:
            if gn: g2p[gn.group(1).lower()]=acc+'|'+pid.group(1)
            if lt: l2p[lt.group(1)]=acc+'|'+pid.group(1)
    acc_gene2prot[acc]=g2p; acc_locus2prot[acc]=l2p
    print(f'{acc}: {len(g2p)} named genes, {len(l2p)} locus tags')

## 5. Map DEG conditional + unconditional essentials -> OG -> conservation

In [ ]:
import csv, re, json, pandas as pd
ogc=json.load(open('outputs/orphan/og_conservation.json'))
COND=re.compile(r'required for|tolerance|model of|infection|sputum|tobramycin|stress|cholesterol|bile|murine|in vivo|host|serum', re.I)
def acc_for(org):
    for sub,a in PATHOGENS.items():
        if sub.lower() in org.lower(): return a
    return None
deg=list(csv.DictReader(open('data/drive_import/deg/deg_bacteria_essential_genes.csv')))
rows=[]
for r in deg:
    a=acc_for(r['organism']);
    if not a: continue
    gn=r['gene_name'].lower() if r['gene_name'] and r['gene_name']!='-' else None
    prot=acc_gene2prot.get(a,{}).get(gn) if gn else None
    if not prot: continue
    og=prot2og.get(prot)
    cons=ogc.get(og,{}).get('conservation') if og else None
    rows.append(dict(organism=r['organism'][:30], gene=gn, condition=r['condition'][:40],
                     conditional=bool(COND.search(r['condition'])), og=og,
                     conservation=cons))
d=pd.DataFrame(rows)
d2=d[d.conservation.notna()]
print(f'DEG pathogen essentials mapped to OG+conservation: {len(d2)} / {len(d)}')
print(d2.conditional.value_counts())

## 6. THE TEST: conditional vs unconditional conservation

In [ ]:
from scipy.stats import mannwhitneyu
import numpy as np, matplotlib.pyplot as plt
c=d2[d2.conditional]; u=d2[~d2.conditional]
print(f'CONDITIONAL essentials:   n={len(c)}  median conservation {c.conservation.median():.3f}  rogue(<0.1) frac {(c.conservation<0.1).mean():.2f}')
print(f'UNCONDITIONAL essentials: n={len(u)}  median conservation {u.conservation.median():.3f}  rogue(<0.1) frac {(u.conservation<0.1).mean():.2f}')
if len(c)>20 and len(u)>20:
    U,p=mannwhitneyu(c.conservation,u.conservation,alternative='less')
    print(f'\nMann-Whitney (conditional < unconditional conservation): p={p:.2e}')
    verdict = ('CONFIRMED: conditional essentials ARE lower-conservation (rogue zone)' if (c.conservation.median()<u.conservation.median() and p<0.05)
               else 'NOT confirmed: conditional essentials are as conserved as unconditional (thesis wrong/nuanced)')
    print('VERDICT:', verdict)
fig,ax=plt.subplots(figsize=(7,5))
ax.hist(u.conservation,bins=30,density=True,alpha=0.5,label=f'unconditional (med {u.conservation.median():.2f})',color='#C0392B')
ax.hist(c.conservation,bins=30,density=True,alpha=0.6,label=f'conditional (med {c.conservation.median():.2f})',color='#16A085')
ax.set_xlabel('conservation (family essentiality fraction)'); ax.set_ylabel('density')
ax.set_title('DEG conditional vs unconditional essentials -- conservation'); ax.legend()
fig.tight_layout(); fig.savefig('outputs/orphan/deg_conditional_conservation.png',dpi=140)
d2.to_csv('outputs/orphan/deg_pathogen_mapped.csv',index=False)
print('\nwrote deg_conditional_conservation.png + deg_pathogen_mapped.csv')
# by condition type
print('\nmedian conservation by condition:')
print(d2.groupby(d2.condition.str.contains('cholesterol|bile|tolerance|host|murine|infection|sputum|tobramycin',case=False)).conservation.median())